# S3Eval: LLM-as-Expert Evaluation (Local Models via Ollama)

This notebook runs LLM-as-expert evaluation using locally hosted models via Ollama.

Models: Qwen3-4B, 8B, 14B, 32B (Q4 quantized)

Rubric variants: simplified (binary) and detailed (scored)

In [11]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


In [12]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    e.g. 'meta/llama-3.3-70b-instruct' -> 'llama-3.3-70b'
         'gpt-4.1-mini' -> 'gpt-4.1-mini'
         'all-MiniLM-L6-v2' -> 'all-MiniLM-L6-v2'
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model.
    GPU for small models (<2B params), CPU for large models.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    # Models known to be too large for GPU encode (>2GB weights)
    FORCE_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B", "jina-embeddings-v3", "stella_en_1.5B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}
    if any(t in model_name for t in FORCE_CPU):
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  Loaded {model_name} on CPU (large model)")
        return model
    try:
        model = SentenceTransformer(model_name, **kwargs)
        print(f"  Loaded {model_name} on {model.device}")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        import gc; gc.collect(); torch.cuda.empty_cache()
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  GPU OOM, loaded {model_name} on CPU")
        return model

def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build 21x55 pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: 21 unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute 21x55 similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


Bad (mutated) examples: 18 CVEs, 55 domains


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Loaded all-MiniLM-L6-v2 on cuda:0


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [13]:
# ── Ollama Client Setup ──
# Ollama serves an OpenAI-compatible API at localhost:11434
ollama_client = OpenAI(
    api_key='ollama',  # required but unused
    base_url='http://localhost:11434/v1',
    timeout=300.0,  # longer timeout for local models
)

# Test connection
try:
    test = ollama_client.chat.completions.create(
        model='qwen3:4b-fp16',
        messages=[{'role': 'user', 'content': 'Say OK'}],
        max_tokens=5,
        temperature=0,
    )
    print(f'Ollama connection OK: {test.choices[0].message.content}')
except Exception as e:
    print(f'Ollama connection FAILED: {e}')
    print('Make sure ollama serve is running!')


Ollama connection OK: 


In [14]:
# ── Override save functions to include scores, llm_response, and reference_ap ──

def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, reference_ap, verdict, final_score, scores, response, parse_success, usage, elapsed, cost):
    result = {
        'timestamp': datetime.now().isoformat(),
        'llm_model': llm_model, 'seed': seed, 'temperature': temperature,
        'source': source, 'cve_id': cve_id, 'ap_id': ap_id,
        'reference_ap': reference_ap,
        'verdict': verdict, 'final_score': final_score,
        'scores': scores,
        'parse_success': parse_success,
        'response_length': len(response),
        'llm_response': response,
        'usage': {**usage, 'elapsed_seconds': elapsed, 'cost_usd': cost},
    }
    with open(results_path, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return result

def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, reference_ap, label, response, parse_success, usage, elapsed, cost):
    result = {
        'timestamp': datetime.now().isoformat(),
        'llm_model': llm_model, 'seed': seed, 'temperature': temperature,
        'source': source, 'cve_id': cve_id, 'ap_id': ap_id,
        'reference_ap': reference_ap,
        'label': label,
        'parse_success': parse_success,
        'response_length': len(response),
        'llm_response': response,
        'usage': {**usage, 'elapsed_seconds': elapsed, 'cost_usd': cost},
    }
    with open(results_path, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return result

print('Override save functions loaded (with scores + llm_response + reference_ap)')


Override save functions loaded (with scores + llm_response + reference_ap)


In [15]:
# ── Response parsing helpers ──
SCORED_CRITERIA = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2']
SCORED_CRITERIA_EXT = SCORED_CRITERIA + ['R1','R2','R3']

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith('```'):
            text = text.split('\n', 1)[1]
            text = text.rsplit('```', 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def parse_binary_response(response_text):
    """Parse binary True/False from LLM response."""
    try:
        text = response_text.strip()
        if text.startswith('```'): text = text.split('\n', 1)[1].rsplit('```', 1)[0]
        raw_label = json.loads(text).get('label', '?')
        return True if str(raw_label).lower() in ('true', '1', 'yes') else False
    except Exception:
        # Fallback: check raw text
        t = response_text.strip().lower()
        if 'true' in t: return True
        if 'false' in t: return False
        return None

print('Parse functions loaded')


Parse functions loaded


In [16]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [17]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [18]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [19]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [20]:
# ── Debug: test with /no_think ──
resp = ollama_client.chat.completions.create(
    model='qwen3:4b-fp16',
    messages=[{'role': 'user', 'content': 'Say True or False: is 1+1=2? /no_think'}],
    max_tokens=50,
    temperature=0,
)
msg = resp.choices[0].message
print('content:', repr(msg.content))
print('finish_reason:', resp.choices[0].finish_reason)

# Also test with eval prompt
print('\n=== Test with actual eval prompt ===')
response, usage = llm_eval_intrinsic_binary(
    dataset[0]['cve_id'], dataset[0]['description'],
    dataset[0]['attack_paths'][0]['domain'],
    ollama_client, 'qwen3:4b-fp16', seed=42, n_calibration=2)
print(f'Response length: {len(response)}')
print(f'Response: {repr(response[:500])}')
print(f'Parsed label: {parse_binary_response(response)}')
print(f'Tokens: {usage}')


content: ''
finish_reason: length

=== Test with actual eval prompt ===
Response length: 15
Response: '{"label": true}'
Parsed label: True
Tokens: {'prompt_tokens': 6494, 'completion_tokens': 1047, 'total_tokens': 7541, 'elapsed_seconds': 11.45}


## qwen3:4b-fp16


In [21]:
# # ── Intrinsic Binary: qwen3:4b-fp16 ──
# MODEL = 'qwen3:4b-fp16'
# save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
# os.makedirs(save_dir, exist_ok=True)
# results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [22]:
# # ── Intrinsic Scored: qwen3:4b-fp16 ──
# MODEL = 'qwen3:4b-fp16'
# results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [23]:
# # ── Extrinsic Binary: qwen3:4b-fp16 ──
# MODEL = 'qwen3:4b-fp16'
# save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
# os.makedirs(save_dir_ext, exist_ok=True)
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [24]:
# # ── Extrinsic Scored: qwen3:4b-fp16 ──
# MODEL = 'qwen3:4b-fp16'
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


## qwen3:8b-fp16


In [25]:
# # ── Intrinsic Binary: qwen3:8b-fp16 ──
# MODEL = 'qwen3:8b-fp16'
# save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
# os.makedirs(save_dir, exist_ok=True)
# results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [26]:
# # ── Intrinsic Scored: qwen3:8b-fp16 ──
# MODEL = 'qwen3:8b-fp16'
# results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [27]:
# # ── Extrinsic Binary: qwen3:8b-fp16 ──
# MODEL = 'qwen3:8b-fp16'
# save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
# os.makedirs(save_dir_ext, exist_ok=True)
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [28]:
# # ── Extrinsic Scored: qwen3:8b-fp16 ──
# MODEL = 'qwen3:8b-fp16'
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


## qwen3:14b


In [ ]:
# # ── Intrinsic Binary: qwen3:14b ──
# MODEL = 'qwen3:14b'
# save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
# os.makedirs(save_dir, exist_ok=True)
# results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


--- Reference (expected: True) ---
  [ref] CVE-2022-1471/AP1  label=True  tokens=7094
  [ref] CVE-2022-40149/AP1  label=True  tokens=4866
  [ref] CVE-2022-40149/AP2  label=True  tokens=4987
  [ref] CVE-2022-40150/AP1  label=True  tokens=4934
  [ref] CVE-2022-40150/AP2  label=True  tokens=5197
  [ref] CVE-2023-2976/AP1  label=True  tokens=5562
  [ref] CVE-2023-2976/AP2  label=True  tokens=5736
  [ref] CVE-2023-2976/AP3  label=True  tokens=5494
  [ref] CVE-2023-33202/AP1  label=True  tokens=7223
  [ref] CVE-2023-33202/AP2  label=False  tokens=7344
  [ref] CVE-2023-33202/AP3  label=True  tokens=6162
  [ref] CVE-2023-34055/AP1  label=False  tokens=4854
  [ref] CVE-2023-44487/AP1  label=True  tokens=7703
  [ref] CVE-2023-46589/AP1  label=True  tokens=10689
  [ref] CVE-2023-46589/AP2  label=True  tokens=10181
  [ref] CVE-2023-46589/AP3  label=True  tokens=9667
  [ref] CVE-2023-46589/AP4  label=True  tokens=9871
  [ref] CVE-2023-6378/AP1  label=True  tokens=5927
  [ref] CVE-2024-12798/AP1  la

In [ ]:
# # ── Intrinsic Scored: qwen3:14b ──
# MODEL = 'qwen3:14b'
# results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


--- Reference (expected: True) ---
  [ref] CVE-2022-1471/AP1  verdict=True  min=5  tokens=9212
  [ref] CVE-2022-40149/AP1  verdict=True  min=5  tokens=6620
  [ref] CVE-2022-40149/AP2  verdict=True  min=4  tokens=6761
  [ref] CVE-2022-40150/AP1  verdict=True  min=5  tokens=6528
  [ref] CVE-2022-40150/AP2  verdict=True  min=5  tokens=6894
  [ref] CVE-2023-2976/AP1  verdict=True  min=5  tokens=7287
  [ref] CVE-2023-2976/AP2  verdict=True  min=5  tokens=7180
  [ref] CVE-2023-2976/AP3  verdict=True  min=5  tokens=7095
  [ref] CVE-2023-33202/AP1  verdict=True  min=5  tokens=8555
  [ref] CVE-2023-33202/AP2  verdict=True  min=5  tokens=7991
  [ref] CVE-2023-33202/AP3  verdict=True  min=5  tokens=7672
  [ref] CVE-2023-34055/AP1  verdict=False  min=1  tokens=6831
  [ref] CVE-2023-44487/AP1  verdict=True  min=5  tokens=9650
  [ref] CVE-2023-46589/AP1  verdict=True  min=4  tokens=11786
  [ref] CVE-2023-46589/AP2  verdict=True  min=5  tokens=11958
  [ref] CVE-2023-46589/AP3  verdict=True  min=5  to

In [ ]:
# # ── Extrinsic Binary: qwen3:14b ──
# MODEL = 'qwen3:14b'
# save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
# os.makedirs(save_dir_ext, exist_ok=True)
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_binary(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
#             save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


--- Reference (self as ref, expected: True) ---
  [ref] CVE-2022-1471/AP1 vs self  label=True
  [ref] CVE-2022-40149/AP1 vs self  label=True
  [ref] CVE-2022-40149/AP2 vs self  label=True
  [ref] CVE-2022-40150/AP1 vs self  label=True
  [ref] CVE-2022-40150/AP2 vs self  label=True
  [ref] CVE-2023-2976/AP1 vs self  label=True
  [ref] CVE-2023-2976/AP2 vs self  label=True
  [ref] CVE-2023-2976/AP3 vs self  label=True
  [ref] CVE-2023-33202/AP1 vs self  label=True
  [ref] CVE-2023-33202/AP2 vs self  label=True
  [ref] CVE-2023-33202/AP3 vs self  label=True
  [ref] CVE-2023-34055/AP1 vs self  label=False
  [ref] CVE-2023-44487/AP1 vs self  label=True
  [ref] CVE-2023-46589/AP1 vs self  label=True
  [ref] CVE-2023-46589/AP2 vs self  label=True
  [ref] CVE-2023-46589/AP3 vs self  label=True
  [ref] CVE-2023-46589/AP4 vs self  label=True
  [ref] CVE-2023-6378/AP1 vs self  label=True
  [ref] CVE-2024-12798/AP1 vs self  label=True
  [ref] CVE-2024-12798/AP2 vs self  label=True
  [ref] CVE-2024

In [ ]:
# # ── Extrinsic Scored: qwen3:14b ──
# MODEL = 'qwen3:14b'
# results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (self as ref, expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (source AP as ref, expected: False) ---')
# for bad_entry in bad_dataset:
#     cve_id = bad_entry['cve_id']
#     ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
#     if ref_entry is None: continue
#     ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
#     for ap in bad_entry['attack_paths']:
#         source_ap = ap['ap_id'].split('_')[0]
#         ref_domain = ref_lookup.get(source_ap)
#         if ref_domain is None:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
#             continue
#         try:
#             response, usage = llm_eval_extrinsic_scored(
#                 cve_id, bad_entry['description'], ref_domain, ap['domain'],
#                 ollama_client, MODEL, seed=42, n_calibration=2)
#             scores = parse_scored_response(response) or {'parse_error': True}
#             final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
#             verdict = True if final_score is not None and final_score >= 3 else False
#             count += 1
#             elapsed = time.time() - t_start
#             print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
#             save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
#         except Exception as e:
#             print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


--- Reference (self as ref, expected: True) ---
  [ref] CVE-2022-1471/AP1 vs self  verdict=True  min=5
  [ref] CVE-2022-40149/AP1 vs self  verdict=True  min=5
  [ref] CVE-2022-40149/AP2 vs self  verdict=True  min=5
  [ref] CVE-2022-40150/AP1 vs self  verdict=True  min=5
  [ref] CVE-2022-40150/AP2 vs self  verdict=True  min=5
  [ref] CVE-2023-2976/AP1 vs self  verdict=True  min=5
  [ref] CVE-2023-2976/AP2 vs self  verdict=True  min=3
  [ref] CVE-2023-2976/AP3 vs self  verdict=True  min=5
  [ref] CVE-2023-33202/AP1 vs self  verdict=True  min=5
  [ref] CVE-2023-33202/AP2 vs self  verdict=True  min=5
  [ref] CVE-2023-33202/AP3 vs self  verdict=True  min=5
  [ref] CVE-2023-34055/AP1 vs self  verdict=True  min=5
  [ref] CVE-2023-44487/AP1 vs self  verdict=True  min=5
  [ref] CVE-2023-46589/AP1 vs self  verdict=True  min=5
  [ref] CVE-2023-46589/AP2 vs self  verdict=True  min=5
  [ref] CVE-2023-46589/AP3 vs self  verdict=True  min=5
  [ref] CVE-2023-46589/AP4 vs self  verdict=True  min=5
  [r

## qwen3:32b


In [ ]:
# # ── Intrinsic Binary: qwen3:32b ──
# MODEL = 'qwen3:32b'
# save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
# os.makedirs(save_dir, exist_ok=True)
# results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
# t_start = time.time()
# count = 0

# print(f'--- Reference (expected: True) ---')
# for entry in dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\n--- Bad (expected: False) ---')
# for entry in bad_dataset:
#     for ap in entry['attack_paths']:
#         try:
#             response, usage = llm_eval_intrinsic_binary(
#                 entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
#             label = parse_binary_response(response)
#             count += 1
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
#             save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
#         except Exception as e:
#             print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

# print(f'\nDone: qwen3:32b | {count} samples | {time.time()-t_start:.1f}s')


--- Reference (expected: True) ---
  [ref] CVE-2022-1471/AP1  label=True  tokens=7028
  [ref] CVE-2022-40149/AP1  label=True  tokens=4791
  [ref] CVE-2022-40149/AP2  label=True  tokens=5084
  [ref] CVE-2022-40150/AP1  label=True  tokens=4705
  [ref] CVE-2022-40150/AP2  label=True  tokens=5182
  [ref] CVE-2023-2976/AP1  label=True  tokens=5746
  [ref] CVE-2023-2976/AP2  label=True  tokens=5551
  [ref] CVE-2023-2976/AP3  label=True  tokens=5247
  [ref] CVE-2023-33202/AP1  label=False  tokens=6905
  [ref] CVE-2023-33202/AP2  label=False  tokens=6412
  [ref] CVE-2023-33202/AP3  label=True  tokens=5855
  [ref] CVE-2023-34055/AP1  label=False  tokens=4915
  [ref] CVE-2023-44487/AP1  label=True  tokens=7537
  [ref] CVE-2023-46589/AP1  label=False  tokens=10136
  [ref] CVE-2023-46589/AP2  label=False  tokens=10139
  [ref] CVE-2023-46589/AP3  label=False  tokens=9436
  [ref] CVE-2023-46589/AP4  label=False  tokens=9942
  [ref] CVE-2023-6378/AP1  label=True  tokens=5859
  [ref] CVE-2024-12798/AP

In [ ]:
# ── Intrinsic Scored: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("ap_id","")))
print(f"Already done: {len(done_keys)}, remaining: {110 - len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        if ("bad", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

elapsed = time.time() - t_start
print(f'\nDone: {count} new results in {elapsed:.1f}s')


In [ ]:
# ── Extrinsic Binary: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("generated","") or d.get("ap_id","")))
print(f"Already done: {len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        if ("bad", cve_id, ap["ap_id"]) in done_keys:
            continue
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:32b | {count} new samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Scored: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("generated","") or d.get("ap_id","")))
print(f"Already done: {len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        if ("bad", cve_id, ap["ap_id"]) in done_keys:
            continue
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:32b | {count} new samples | {time.time()-t_start:.1f}s')
